# Introduction to Profiling — GNU gprof
## Concepts and practice · NCI ARE · 12 cores on one node


We use **prebuilt LULESH executables** from the private `lulesh-gprof` module for `vp91`.
By the end you should be able to collect `gmon` data, distinguish sampled timing from call counting, read flat profiles and call graphs, measure instrumentation overhead, and explain which questions need another profiler.


In [ ]:
import os
# the jupyter notebook is launched from your $HOME, change the working directory provided a username directory is created under /scratch/vp91
os.chdir(os.path.expandvars("/scratch/vp91/$USER/intro-to-profiling"))


## Exercise roadmap

| Stop | Time | Task | Evidence to retain |
|---|---:|---|---|
| [Setup](#setup) | 5–10 min | ARE allocation and private module | Executable identities and environment |
| [1. Establish a baseline](#exercise-1) | 8 min | Warm up; repeat the uninstrumented executable | Three times and numerical diagnostics |
| [2. Collect and read a flat profile](#exercise-2) | 8 min | Run the `-pg` executable; inspect raw data | Raw file, report and expensive functions |
| [3. Follow the call graph](#exercise-3) | 8 min | Distinguish self cost from estimated descendant cost | One caller/function/child explanation |
| [4. Measure profiling overhead](#exercise-4) | 10 min | Alternate matched baseline and instrumented runs | Repeated times and observed overhead |
| [5. Check repeatability](#exercise-5) | 11 min | Compare repeats, then sum compatible profiles | Stable observations and a testable hypothesis |
| [MPI extension](#mpi-extension) | Optional | Eight single-threaded ranks | Eight rank-identified files and individual reports |



## What does gprof measure?

Gprof combines two mechanisms:

- **Sampling:** periodic observations of the program counter estimate time spent in code. 
- **Instrumentation:** compiling with `-pg` adds entry hooks that count observed caller–callee arcs. These hooks do not time every call from entry to exit. Optimisation can inline functions or change call boundaries.

**Predict:** a function executes a million times but does almost no work per call. Another runs ten times and does expensive calculations. Which could dominate?


### Two matching executables

| Executable | Use | Configuration |
|---|---|---|
| `lulesh2.0-gprof-baseline` | Unprofiled timing | `-O3 -g -DNDEBUG -fno-omit-frame-pointer` |
| `lulesh2.0-gprof` | Collect gprof data | Same options, plus **`-pg` at compilation and linking** |

Both builds use the same source, GCC/Open MPI versions, MPI enabled and OpenMP disabled. Note that Debug symbols (`-g`) and gprof instrumentation (`-pg`) have different purposes.




The next cell finds both executables on `PATH`. Results are written to `/scratch/vp91/<your-user>/gprof-results/`. The shared application/source installation is used for reading and execution.

In [ ]:
from pathlib import Path
import getpass
import json
import os
import shutil
import statistics
import subprocess
import sys

WORK = Path(f"/scratch/vp91/{getpass.getuser()}")
candidates = [Path.cwd(), Path.cwd() / "gprof", *[p / "gprof" for p in Path.cwd().parents],
              WORK / "intro-to-profiling" / "gprof"]
LAB_DIR = next((p for p in candidates if (p / "gprof_lab.py").is_file()), None)
if LAB_DIR is None:
    raise RuntimeError("Set LAB_DIR to the training checkout's gprof directory")
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))
from gprof_lab import Lab, check_session, merge_profiles, report, sha256

required = ("lulesh2.0-gprof-baseline", "lulesh2.0-gprof", "mpirun", "gprof")
missing = [name for name in required if shutil.which(name) is None]
if missing:
    raise RuntimeError(f"Missing {missing}; configure lulesh-gprof in ARE Modules and relaunch")
BASELINE = Path(shutil.which("lulesh2.0-gprof-baseline")).resolve()
INSTRUMENTED = Path(shutil.which("lulesh2.0-gprof")).resolve()
source = os.environ.get("LULESH_SOURCE")
LULESH_SOURCE = Path(source).resolve() if source else None
RESULTS = WORK / "gprof-results"
ALLOCATED_CORES = 12
lab = Lab(BASELINE, INSTRUMENTED, RESULTS, allocated_cores=ALLOCATED_CORES,
          ranks=1, size=30, iterations=200)
print("Baseline:", BASELINE)
print("Instrumented:", INSTRUMENTED)
print("Source:", LULESH_SOURCE)
print("Results:", RESULTS)

In [ ]:
session = check_session(ALLOCATED_CORES)
lab.validate()
versions = {}
for tool in ("mpirun", "gprof"):
    completed = subprocess.run([tool, "--version"], text=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, check=True)
    versions[tool] = completed.stdout
    print(completed.stdout.splitlines()[0])
if not any(name in versions["mpirun"] for name in ("Open MPI", "OpenRTE")):
    raise RuntimeError("The lesson's binding options and rank wrapper require Open MPI")
RESULTS.mkdir(parents=True, exist_ok=True)
environment = {"session": session, "versions": versions,
               "modules": os.environ.get("LOADEDMODULES"), "source": str(LULESH_SOURCE),
               "baseline": str(BASELINE), "baseline_sha256": sha256(BASELINE),
               "instrumented": str(INSTRUMENTED), "instrumented_sha256": sha256(INSTRUMENTED)}
# Run metadata also retains the environment and binary hash for each execution.
(RESULTS / "environment.json").write_text(json.dumps(environment, indent=2))
manifest = os.environ.get("LULESH_GPROF_BUILD_INFO")
if manifest and Path(manifest).is_file():
    print(Path(manifest).read_text())
else:
    print("Ask the instructor for the matching build flags and source revision.")
if LULESH_SOURCE is None or not LULESH_SOURCE.is_dir():
    print("Source inspection is optional until the instructor supplies the matching source path.")

### Choose a useful workload

LULESH is a hydrodynamics code. `-s N` gives the **local** mesh side length; total elements are `ranks × N³`. `-i K` caps iterations. 

Start at `-s 30 -i 200`. Aim for several seconds in LULESH's reported calculation time, then verify there are enough samples. If the run is too short, increase `lab.size` or `lab.iterations`, rerun the warm-up and establish a fresh baseline.

<a id="exercise-1"></a>
## Exercise 1 — Establish a baseline (8 min)

Start with the baseline executable. Inspect the log for the number of ranks, actual iterations and successful completion. 

In [ ]:
warmup = lab.run(label="baseline-warmup")
print("LULESH calculation seconds:", warmup["application_seconds"])
print("Launcher-to-exit seconds:", warmup["launcher_wall_seconds"])

The helper saves both **LULESH's own elapsed time** and **launcher-to-exit wall time**. These measure different intervals. We use the same LULESH timing field throughout the timing exercises. The launcher time additionally includes process startup and, for profiling runs, profile-file writing.

Check that the workload lasts long enough before running the next cell. Run all experiments sequentially within this allocation.

In [ ]:
baselines = [lab.run(label=f"baseline-{i+1}") for i in range(3)]
def application_times(records):
    values = [r["application_seconds"] for r in records]
    if any(v is None or v <= 0 for v in values):
        raise RuntimeError("Read run.log: LULESH calculation time was absent or nonpositive")
    return values
baseline_times = application_times(baselines)
print("Calculation seconds:", baseline_times)
print("Median:", statistics.median(baseline_times))
print("Range:", min(baseline_times), "to", max(baseline_times))

**Checkpoint:** record three times, the timing interval, input and numerical diagnostics. 

## From executable to report

`gprof` is not a launcher, it only reads data **after** the application exits. An equivalent one-rank collection in a fresh writable directory is:

```bash
export OMP_NUM_THREADS=1 OMP_DYNAMIC=FALSE
export GMON_OUT_PREFIX="$PWD/gmon"
mpirun -np 1 --map-by slot:PE=1 --bind-to core \
  lulesh2.0-gprof -s 30 -i 200
# Use the actual PID-suffixed file written by this run:
gprof -b --demangle -p /path/to/lulesh2.0-gprof /path/to/gmon.PID
```

The helper starts each rank with `rank_run.py`, records its rank/PID and sets an absolute prefix `gmon.rank-000`, etc. `exec` preserves that PID when replacing the wrapper with LULESH. Raw files, commands, hashes and logs remain together in a fresh personal directory.

<a id="exercise-2"></a>
## Exercise 2 — Collect and read a flat profile (8 min)

**Predict:** will the function with the most calls also have the greatest self time?

Run the instrumented executable using exactly the same mesh, iteration cap, rank count and placement as the baseline. Check numerical output before interpreting costs.

In [ ]:
profile_run = lab.run(profile=True, label="first-profile")
print("Raw profile files:", *profile_run["profiles"], sep="\n")
info_path = report(profile_run, "info")
flat_path = report(profile_run, "flat", lines=35)

### Read the units and denominator

| Field | Interpretation |
|---|---|
| `% time` | Share of the report's attributed sampled time spent in the function itself |
| `self seconds` | Sampled time attributed to that function's own code |
| `cumulative seconds` | Running sum down the flat table; not descendants of this function |
| `calls` | Recorded instrumented calls where available|
| `self .../call` | Self time divided by the recorded calls |
| `total .../call` | Self plus estimated descendant time per call |


**Illustrative arithmetic:** 0.8 self seconds over 400 calls gives 2 ms of average self time per call. If the report attributes 2 seconds in total, this function has 40% self time. 

`-b` removes explanatory prose; `-p` selects the flat profile; `--demangle` requests readable C++ names. `-C` means execution counts in GNU gprof


**Checkpoint:** record the sampling period, three costly functions, their self time and calls.

<a id="exercise-3"></a>
## Exercise 3 — Follow the call graph 

**Predict:** can a function have almost no self time but a large call-graph percentage?

In each entry, the primary function row starts with `[index]`. Its callers appear above it and its children below. Primary-row `% time` includes **self plus estimated descendants**. Costs overlap across parent/child entries, so these percentages should not be added.

Gprof distributes a callee's cost among callers using call counts. It assumes comparable average cost per call from each caller. If one caller triggers expensive work and another mostly takes an early return, that attribution may mislead. This is not a measured entry-to-exit duration for each call path.



In [ ]:
graph_path = report(profile_run, "graph", lines=70)
# For the complete graph, open graph_path in JupyterLab or use:
# print(graph_path.read_text())

In [ ]:
FUNCTION_NAME = "CalcForceForNodes"  # EDIT: choose a function actually present in your report
print("Report lines containing", FUNCTION_NAME)
for line in graph_path.read_text().splitlines():
    if FUNCTION_NAME in line:
        print(line)
if LULESH_SOURCE is None or not LULESH_SOURCE.is_dir():
    print("Obtain the matching source directory from the instructor; use the graph meanwhile.")
else:
    matches = 0
    for path in sorted(LULESH_SOURCE.glob("*.cc")):
        for number, line in enumerate(path.read_text(errors="replace").splitlines(), 1):
            if FUNCTION_NAME in line:
                print(f"{path}:{number}: {line.strip()}")
                matches += 1
    if not matches:
        print("Try the unmangled base name of a function visible in your report.")

**Checkpoint:** identify one function, a caller and a child; explain whether its cost lies mostly in self or descendants. Inspect the source to propose a reason, without treating a source-text match as per-line timing.

**Predict:** function `step()` is credited with 70% including children, and its child `forces()` with 60%. Is their combined contribution 130%? Explain the overlap.


<a id="exercise-4"></a>
## Exercise 4 — Measure profiling overhead (10 min)

Warm up the instrumented build, then alternate the order of three baseline/profile pairs. Use the **matched pair built with OpenMP disabled**, not the existing VTune executable. Keep every other setting fixed and inspect all numerical logs.

Compute the observed relative timing difference as `100 × (median(T_pg) / median(T_base) − 1)`. It measures the effect of this instrumented build for this workload and timing region; it is not a universal gprof overhead percentage. The value may be noisy or even negative. Report the spread rather than forcing a positive answer.

In [ ]:
_ = lab.run(profile=True, label="profile-warmup")
paired = {"baseline": [], "profile": []}
for repetition in range(3):
    order = [False, True] if repetition % 2 == 0 else [True, False]
    for instrument in order:
        record = lab.run(profile=instrument, label=f"pair-{repetition+1}")
        paired[record["mode"]].append(record)
base_times = application_times(paired["baseline"])
pg_times = application_times(paired["profile"])
print("Baseline calculation seconds:", base_times)
print("Instrumented calculation seconds:", pg_times)
observed_overhead = 100 * (statistics.median(pg_times) / statistics.median(base_times) - 1)
print(f"Observed relative median difference: {observed_overhead:.1f}%")
print("Check numerical diagnostics and timing ranges before interpreting this difference.")

**Checkpoint:** record both medians and ranges, the timing interval, the two binary hashes and the observed difference. A faster instrumented observation amid overlapping timings is not evidence that instrumentation improves the application. To measure an optimisation later, compare repeated **uninstrumented** builds and check correctness.

<a id="exercise-5"></a>
## Exercise 5 — Check repeatability and form a hypothesis (11 min)

Compare the three instrumented runs from Exercise 4. Look at **absolute self times and calls**, as well as percentages. Is the top cost stable? Are apparent ties larger than a sample period?

Summing compatible profiles adds their sample and call counts. It can reduce sampling noise, but does not remove instrumentation bias or the call-graph attribution assumptions. It is a **sum, not an average**. Only combine the same executable and comparable workload; verify actual iterations and numerical diagnostics before merging.

In [ ]:
for record in paired["profile"]:
    report(record, "flat", lines=22)

**Predict before running:** if each of three deterministic runs records 400 calls to a function, how many calls should their merged report show?

The helper checks binary hashes, input parameters and process/thread counts and rejects duplicate input files. You must still check the actual iteration counts and correctness output in the logs.

In [ ]:
merged_directory = merge_profiles(paired["profile"], RESULTS)

**Checkpoint:** explain the sum, then complete a statement using your own results:

> In this workload, function ___ repeatedly accounts for ___ sampled self seconds. Its source suggests ___. I would test ___ while holding ___ fixed, check numerical correctness, and compare repeated uninstrumented elapsed times.

If the question concerns cache misses, OpenMP worker threads, blocked time or MPI communication, state the missing evidence and choose a tool that measures it. A costly function does not by itself establish a cache, bandwidth or communication bottleneck.

<a id="mpi-extension"></a>
## Optional MPI extension — Eight ranks in the same ARE session

Use **8 ranks × 1 application thread**, leaving the allocation at 12 cores. Twelve MPI ranks are invalid for LULESH. The helper gives every rank a distinct prefix, keeps a rank/PID/hostname manifest and checks that all eight profiles were written.

A PID is not an MPI rank. Inspect individual rank reports before an aggregate. An all-rank report sums attributed CPU costs and calls; it is not job wall time, load-balance efficiency or a measurement of all MPI communication. A rank with less sampled compute time might have waited longer.

Holding `-s 30` fixed multiplies the global element count by eight relative to the one-rank run. This is a **profile-coverage exercise**, not a fixed-work speedup experiment.

Set `RUN_MPI_EXTENSION=True` when the instructor enables this extension. [LLNL gprof MPI examples](https://hpc.llnl.gov/software/development-environment-software/gprof).

In [ ]:
RUN_MPI_EXTENSION = False
if RUN_MPI_EXTENSION:
    mpi_lab = Lab(BASELINE, INSTRUMENTED, RESULTS, allocated_cores=ALLOCATED_CORES,
                  ranks=8, size=lab.size, iterations=lab.iterations)
    mpi_run = mpi_lab.run(profile=True, label="eight-rank-profile")
    for rank in range(8):
        report(mpi_run, "flat", rank=rank, lines=15)
    report(mpi_run, "flat", rank=None, lines=30)
    print("Inspect rank-*.json files for the rank/PID mapping; compare each rank before the sum.")
else:
    print("Optional MPI extension skipped. The five core exercises are complete.")

<a id="worksheet"></a>
## Results worksheet

ARE job / hostname: …  Allocation: **12 cores, one node**

Module versions / build manifests / source revision: …

Baseline binary hash: …  Instrumented binary hash: …

Mesh / iteration cap / actual iterations / ranks: …

Numerical checks and tolerances: …  Timing interval: …

| Observation | Measured value | Saved evidence | Interpretation / limitation |
|---|---|---|---|
| Baseline median and range | | | |
| Instrumented median and range | | | |
| Observed relative timing difference | | | |
| Sampling period / top self cost | | | |
| Call count and graph descendants | | | |
| Repeat stability / merged sum | | | |
| Proposed change and validation | | | |
| Optional eight-rank coverage | | | |

## Troubleshooting

| Symptom | Check |
|---|---|
| Missing application or `gprof` | ARE module directory, `lulesh-gprof`, runtime dependencies and mounted storage; relaunch the kernel environment |
| Wrong allocation or multiple nodes | Submit ARE with 12 cores on one node |
| No profile after success | Correct `-pg` binary, `-pg` at compile/link, glibc prefix support, writable output directory and normal termination |
| Only `gmon.out` appears | Prefix support/environment; do not run multiple ranks into the same default filename |
| Missing call-graph records | Check all required application objects were compiled with `-pg` and the executable linked with `-pg` |
| Nearly all sampled times are zero | Increase representative work or combine compatible repeated profiles; examine the stated sample period |
| Many missing calls or functions | Inlining, uninstrumented libraries, sampling resolution and tool scope; do not interpret blanks as zero |
| Mangled C++ names | Use `--demangle`; `-C` requests execution counts |
| Binary hash mismatch | Restore the exact executable used for collection; do not interpret old data with a rebuild |
| Baseline writes profiling data | The baseline is instrumented; fix the module/build pair |
| No source matches | Use the source revision used to build and the base function name; source inspection can wait |
| Missing MPI ranks | Check exit status, per-rank manifests, file naming and storage; do not silently report a partial collection |

Every failed application run keeps `run.log` and `run.json`. A failed report retains its diagnostic output. Fix collection errors before making timing claims.

## Check your understanding

1. Which parts of gprof's evidence come from counting and which from sampling?
2. Why is `-g` insufficient to generate a classic gprof call graph?
3. Why can flat-profile cumulative seconds differ from time including children?
4. Why should call-graph percentages not be added across parent and child?
5. What assumption is used when a callee has several callers?
6. Why keep an executable without `-pg` for timing?
7. Why does this lesson disable OpenMP even though ARE has 12 cores?
8. Does merging eight rank profiles give the MPI job's elapsed time?

<details><summary>Reference answers — open after discussing</summary>

1. Instrumentation counts observed call arcs; sampling estimates self time. Descendant time is inferred.
2. `-g` supplies debug information. `-pg` is needed during compilation and linking to arrange instrumentation and collection.
3. Cumulative seconds adds preceding flat-table rows; it does not follow the call graph.
4. Descendant costs are already included in ancestors, so costs overlap.
5. Average cost per call is assumed independent of the caller; input-dependent work can violate this.
6. Instrumentation affects the executable and execution cost. Improvements need repeated uninstrumented timings.
7. Classic gprof's counting/sampling has threading limitations; the main work uses one application thread per process. Core allocation is a resource limit, not a requirement to run twelve workers.
8. No. It sums recorded CPU costs and calls; blocked time and communication are not fully characterised.

</details>

## References and provenance

The lesson adapts the repository's Gprof RST tutorial and the MPI workshop's concept/exercise/checkpoint style. Slides use the supplied NCI MPI Beamer template, following the VTune deck. The Gprof lesson corrects the older RST's demangling flag and inclusive call-graph percentage definition.

- [GNU gprof manual](https://sourceware.org/binutils/docs/gprof/): compiling, execution, flat profiles, call graphs, sampling error and child-time assumptions.
- [LLNL LULESH](https://github.com/LLNL/LULESH): application source and MPI decomposition.
- [LLNL gprof guide](https://hpc.llnl.gov/software/development-environment-software/gprof): process profiles and aggregation.
- [NCI JupyterLab training guide](https://nci900-training-organisation.github.io/learning-resources/Getting-Started/use-jupyterlab.html): ARE launch environment.

The module and runtime must be verified in a participant ARE session. Local helper tests validate orchestration and GNU gprof behaviour on a small fixture; they do not establish Gadi timings or LULESH numerical correctness.